In [1]:
import dotenv

import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterstats
import exactextract

from rasterstats import zonal_stats
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.mask import mask

from food_security import salinity_correction, water_quality

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / "OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/"

In [3]:
com_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
com_gdf = com_gdf.to_crs("EPSG:4326")

In [4]:
excel_path = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt_ERF_data/data_correlation.xlsx"

def write_excel_file(df, excel_path, sheet_name, append=False):
    # Open Excel file
    try:
        # book = load_workbook(excel_path)
        with pd.ExcelWriter(
            excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
        ) as writer:
            # excel_file.book = book
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except Exception as e:
        print(e)
        df.to_excel(excel_path, sheet_name=sheet_name, index=False)

In [5]:
population_file = src_dir / "egy_pop_2022_CN_100m_R2025A_v1.tif"
classification_file = src_dir / "EGY_DUG_2022_GRID_L1_R2025A_v1.tif"

with rasterio.open(population_file) as pop_src:
    population_shape = pop_src.shape
    transform = pop_src.transform
    crs = pop_src.crs

with rasterio.open(classification_file) as cls_src:
    classification = np.empty(population_shape, dtype=np.uint8)

    reproject(
        source=rasterio.band(cls_src, 1),
        destination=classification,
        src_transform=cls_src.transform,
        src_crs=cls_src.crs,
        dst_transform=transform,
        dst_crs=crs,
        dst_shape=population_shape,
        resampling=Resampling.nearest,
    )

with rasterio.open(population_file) as pop_src:
    population = pop_src.read(1)

    stats = zonal_stats(
        com_gdf,
        population,
        affine=pop_src.transform,
        stats=["sum"],
        nodata=pop_src.nodata,
    )

com_gdf["population"] = [s["sum"] for s in stats]

with rasterio.open(population_file) as pop_src:
    population = pop_src.read(1)
    
    masked_pop = np.where(classification == 1, population, pop_src.nodata)
    
    stats = zonal_stats(
        com_gdf,
        masked_pop,
        affine=pop_src.transform,
        stats=["sum"],
        nodata=pop_src.nodata,
    )

com_gdf["rural_population"] = [s["sum"] for s in stats]

/var/folders/1z/6xp0rndx2nqc7qx263rv42fh0000gn/T/ipykernel_39463/775853396.py:24: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  population = pop_src.read(1)
/var/folders/1z/6xp0rndx2nqc7qx263rv42fh0000gn/T/ipykernel_39463/775853396.py:37: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  population = pop_src.read(1)


In [6]:
output_file = src_dir / "egy_rural_population.tif"

with rasterio.open(population_file) as pop_src:
    population = pop_src.read(1)

    masked_pop = np.where(
        classification == 1,
        population,
        pop_src.nodata,
    )

    profile = pop_src.profile.copy()

    with rasterio.open(output_file, "w", **profile) as dst:
        dst.write(masked_pop.astype(np.float32), 1)

/var/folders/1z/6xp0rndx2nqc7qx263rv42fh0000gn/T/ipykernel_39463/4079373543.py:4: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  population = pop_src.read(1)


In [7]:
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling

gdp_file = src_dir / 'rast_adm2_gdp_perCapita_1990_2022.tif'
rural_pop_file = src_dir / "egy_rural_population.tif"

with rasterio.open(population_file) as pop_src:
    pop = pop_src.read(1).astype("float64")
    pop_profile = pop_src.profile
    pop_transform = pop_src.transform
    pop_crs = pop_src.crs
    pop_shape = pop.shape
    pop_nodata = pop_src.nodata

with rasterio.open(gdp_file) as gdp_src:
    gdp_pc_resampled = np.full(
        pop_shape,
        np.nan,
        dtype="float64",
    )

    reproject(
        source=rasterio.band(gdp_src, 33),
        destination=gdp_pc_resampled,
        src_transform=gdp_src.transform,
        src_crs=gdp_src.crs,
        src_nodata=gdp_src.nodata,
        dst_transform=pop_transform,
        dst_crs=pop_crs,
        dst_nodata=np.nan,
        resampling=Resampling.nearest,
    )

/var/folders/1z/6xp0rndx2nqc7qx263rv42fh0000gn/T/ipykernel_39463/3589392094.py:10: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  pop = pop_src.read(1).astype("float64")


In [8]:
if pop_nodata is not None:
    pop = np.where(pop == pop_nodata, np.nan, pop)

gdp_total = gdp_pc_resampled * pop

In [9]:
stats = zonal_stats(
        com_gdf,
        gdp_total,
        affine=pop_transform,
        stats=["sum"],
        nodata=np.nan,
    )

com_gdf["gdp"] = [s["sum"] for s in stats]
com_gdf['gdp_pc'] = com_gdf['gdp'] / com_gdf['population']

In [10]:
# files
rwi_file = src_dir / "egypt_relative_wealth_index.csv"

# read RWI points
rwi_df = pd.read_csv(rwi_file)

rwi_gdf = gpd.GeoDataFrame(
    rwi_df,
    geometry=gpd.points_from_xy(rwi_df["longitude"], rwi_df["latitude"]),
    crs="EPSG:4326",
)

# spatial join: assign each point to a province
rwi_joined = gpd.sjoin(
    rwi_gdf,
    com_gdf[["geometry", "Name"]],
    how="inner",
    predicate="within",
)

# aggregate to province
com_rwi = (
    rwi_joined
    .groupby("Name")["rwi"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

# join back to province polygons
com_gdf = com_gdf.merge(
    com_rwi,
    on="Name",
    how="left",
)

com_gdf = com_gdf.rename(columns={"mean": "rwi_mean", "median": "rwi_median", "count": "rwi_count"})

In [11]:
command_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
conversion_df = pd.read_excel(excel_path, sheet_name='command_area')

mapping = (
    command_gdf.merge(
        conversion_df[['area_map_name', 'area_name']],
        left_on='OBJECTID',
        right_on='area_map_name'
    )
    .set_index('Name')['area_name']
)

com_gdf["Name"] = com_gdf["Name"].map(mapping)

In [12]:
excel_df = pd.DataFrame(
    {
        "area": com_gdf["Name"],
        "population": com_gdf["population"],
        "rural_population": com_gdf["rural_population"],
        "gdp": com_gdf["gdp"],
        "gdp_pc": com_gdf["gdp_pc"],
        "rwi_mean": com_gdf["rwi_mean"],
        "rwi_median": com_gdf["rwi_median"],
        "rwi_count": com_gdf["rwi_count"],
    }
)

write_excel_file(excel_df, excel_path=excel_path, sheet_name='gdp')

In [13]:
excel_command_df = pd.read_excel(excel_path, sheet_name='command_unit')

excel_command_df = (
    excel_command_df
    .drop(columns=excel_df.columns.difference(["area"]), errors="ignore")
    .merge(excel_df, on="area", how="left")
)

write_excel_file(excel_command_df, excel_path, sheet_name='command_unit')